# Unit 4: EEG 信号特征提取与分析

## 学习目标

- 掌握频带功率 (δ/θ/α/β/γ) 的计算与解释
- 理解事件相关去同步/同步 (ERD/ERS) 的概念和量化方法
- 掌握共空间模式 (CSP) 特征提取算法
- 能够进行多通道 EEG 的 PSD 地形图分析
- 实现基本的 EEG 统计分析与可视化

---

## 4.1 频带功率分析

频带功率是 EEG 分析中最基本的特征。通过计算信号在各经典频段内的能量，可以推断大脑的功能状态。

### 功率谱密度 (PSD) 估计方法

| 方法 | 原理 | 优缺点 |
|------|------|--------|
| **Welch** | 分段加窗 FFT + 平均 | 方差小，分辨率固定 |
| **Multitaper** | 多 Slepian 窗 + 平均 | 最佳偏-方差权衡 |
| **Burg/AR** | 自回归模型 | 短数据效果好，需选择模型阶数 |

### 频带绝对功率与相对功率

- **绝对功率**：频段内的 PSD 积分，单位 μV²
- **相对功率**：频段功率 / 总功率，无量纲
- **相对功率** 对个体差异更稳健

In [ ]:
# ============================================================
# 代码 4.1: 频带功率分析
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import sys
sys.path.insert(0, "..")
from utils.helpers import generate_synthetic_eeg, compute_band_powers, BAND_COLORS

SFREQ = 250.0

# 生成两种状态的数据：闭眼 (强 alpha) 和睁眼 (弱 alpha)
print("Generating two conditions: Eyes Closed vs Eyes Open\n")

# 状态 1: 闭眼放松 (强 alpha)
eyes_closed, _ = generate_synthetic_eeg(
    duration=10.0, sfreq=SFREQ, n_channels=8,
    noise_level=5.0, alpha_power=50.0, seed=100
)

# 状态 2: 睁眼专注 (弱 alpha, 强 beta)
eyes_open, _ = generate_synthetic_eeg(
    duration=10.0, sfreq=SFREQ, n_channels=8,
    noise_level=5.0, alpha_power=10.0, seed=200
)
# 手动增强 beta
t = np.arange(eyes_open.shape[1]) / SFREQ
for ch in range(8):
    eyes_open[ch] += 15 * np.sin(2 * np.pi * 20 * t)  # beta 20Hz

# 计算频带功率
bands = {"delta": (0.5, 4), "theta": (4, 8), "alpha": (8, 13),
         "beta": (13, 30), "gamma": (30, 45)}

powers_closed = compute_band_powers(eyes_closed, SFREQ, bands)
powers_open = compute_band_powers(eyes_open, SFREQ, bands)

# 可视化对比
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 柱状图：绝对功率
x = np.arange(len(bands))
width = 0.35

closed_abs = [powers_closed[b]["absolute"] for b in bands]
open_abs = [powers_open[b]["absolute"] for b in bands]

axes[0].bar(x - width/2, closed_abs, width, label="Eyes Closed",
            color="steelblue", alpha=0.8)
axes[0].bar(x + width/2, open_abs, width, label="Eyes Open",
            color="coral", alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(bands.keys())
axes[0].set_ylabel("Absolute Power (muV^2)")
axes[0].set_title("Absolute Band Powers")
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis="y")

# 柱状图：相对功率
closed_rel = [powers_closed[b]["relative"] * 100 for b in bands]
open_rel = [powers_open[b]["relative"] * 100 for b in bands]

axes[1].bar(x - width/2, closed_rel, width, label="Eyes Closed",
            color="steelblue", alpha=0.8)
axes[1].bar(x + width/2, open_rel, width, label="Eyes Open",
            color="coral", alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(bands.keys())
axes[1].set_ylabel("Relative Power (%)")
axes[1].set_title("Relative Band Powers")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

plt.suptitle("EEG Band Power: Eyes Closed vs Eyes Open",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n频带功率分析结果:")
print(f"{'Band':<8} {'Closed (abs)':<15} {'Open (abs)':<15} {'Closed (rel%)':<15} {'Open (rel%)':<15}")
print("-" * 68)
for band in bands:
    print(f"{band:<8} {powers_closed[band]['absolute']:<15.4f} "
          f"{powers_open[band]['absolute']:<15.4f} "
          f"{powers_closed[band]['relative']*100:<15.2f} "
          f"{powers_open[band]['relative']*100:<15.2f}")

print("\n关键观察:")
print("- 闭眼时 alpha 相对功率显著高于睁眼 (alpha 阻断效应)")
print("- 睁眼时 beta 相对功率升高 (视觉加工激活)")

## 4.2 事件相关去同步/同步 (ERD/ERS)

ERD/ERS 量化了特定频段能量相对于基线期的变化：

$$\text{ERD/ERS}(t) = \frac{P_{\text{active}}(t) - P_{\text{baseline}}}{P_{\text{baseline}}} \times 100\%$$

- **ERD (负值)**：事件后功率下降 → 皮层激活（去同步）
- **ERS (正值)**：事件后功率上升 → 皮层抑制（同步化）

### 经典 ERD/ERS 范式

- **运动想象 (MI)**：对侧感觉运动区 α/β ERD
- **视觉刺激**：枕叶 α ERD
- **记忆编码**：额叶 θ ERS、枕叶 α ERD

In [ ]:
# ============================================================
# 代码 4.2: ERD/ERS 分析演示
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.ndimage import gaussian_filter1d

SFREQ = 250.0

# 模拟运动想象 ERD
# 假设 C3 通道，基线期 [-2, 0]s, 运动想象期 [0, 4]s
t_total = 6.0  # 总时长: -2s to 4s
t_event = 2.0  # 事件时刻 (t=0 in epoch)
n_samples = int(t_total * SFREQ)
t_epoch = np.linspace(-2, 4, n_samples)

# 模拟 alpha (10Hz) 振荡，事件后功率下降 50%
rng = np.random.default_rng(42)
alpha_base_amp = 30.0
alpha_active_amp = 15.0  # 下降 50%
noise_level = 5.0

signal_erd = np.zeros(n_samples)
for i, t in enumerate(t_epoch):
    if t < 0:
        amp = alpha_base_amp
    else:
        amp = alpha_active_amp
    signal_erd[i] = amp * np.sin(2 * np.pi * 10 * t) + rng.normal(0, noise_level)

# 短时傅里叶变换计算时频功率
nperseg = int(0.5 * SFREQ)  # 500ms 窗口
noverlap = int(0.4 * SFREQ)
f_stft, t_stft, Zxx = signal.stft(signal_erd, fs=SFREQ, nperseg=nperseg,
                                    noverlap=noverlap, nfft=512)

# 提取 alpha 频段 (8-13 Hz) 功率
alpha_mask = (f_stft >= 8) & (f_stft <= 13)
alpha_power_t = np.mean(np.abs(Zxx[alpha_mask, :])**2, axis=0)
t_stft_aligned = t_stft - t_event  # 对齐到事件

# 计算 ERD/ERS (%)
baseline_mask = t_stft_aligned < 0
baseline_power = np.mean(alpha_power_t[baseline_mask])
erd_ers = (alpha_power_t - baseline_power) / baseline_power * 100

# 可视化
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# 1. 时域信号
axes[0].plot(t_epoch, signal_erd, color="steelblue", linewidth=0.5)
axes[0].axvline(0, color="red", linestyle="--", linewidth=2, label="Event")
axes[0].axvspan(-2, 0, alpha=0.1, color="green", label="Baseline")
axes[0].axvspan(0, 4, alpha=0.1, color="orange", label="Active")
axes[0].set_ylabel("Amplitude (muV)")
axes[0].set_title("Simulated Motor Imagery ERD Signal (C3 channel)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. 时频图
im = axes[1].pcolormesh(t_epoch, f_stft[f_stft <= 50],
                         np.abs(Zxx[f_stft <= 50, :]),
                         shading="gouraud", cmap="viridis")
axes[1].axvline(0, color="red", linestyle="--", linewidth=2)
axes[1].set_ylabel("Frequency (Hz)")
axes[1].set_title("Time-Frequency Representation")
plt.colorbar(im, ax=axes[1], label="Magnitude")

# 3. ERD/ERS 曲线
axes[2].fill_between(t_stft_aligned, 0, erd_ers,
                      where=(erd_ers >= 0), color="red", alpha=0.5,
                      label="ERS (sync)")
axes[2].fill_between(t_stft_aligned, 0, erd_ers,
                      where=(erd_ers < 0), color="blue", alpha=0.5,
                      label="ERD (desync)")
axes[2].axhline(0, color="black", linestyle="-", linewidth=1)
axes[2].axvline(0, color="red", linestyle="--", linewidth=2)
axes[2].set_xlabel("Time (s)")
axes[2].set_ylabel("ERD/ERS (%)")
axes[2].set_title(f"Alpha Band (8-13 Hz) ERD/ERS (baseline = {baseline_power:.1f})")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"基线期 alpha 功率: {baseline_power:.2f}")
print(f"活动期平均 ERD:     {np.mean(erd_ers[t_stft_aligned > 0]):.1f}%")

## 4.3 共空间模式 (CSP) — BCI 的核心特征算法

CSP 是运动想象 BCI 中最经典的空域滤波算法。其目标是找到一个投影矩阵 $W$，使得：

- 投影后，一类信号的方差最大化，另一类最小化

### 数学原理

对于两类协方差矩阵 $\Sigma_1$ 和 $\Sigma_2$：

$$w^* = \arg\max_w \frac{w^T \Sigma_1 w}{w^T (\Sigma_1 + \Sigma_2) w}$$

这等价于解广义特征值问题 $\Sigma_1 w = \lambda (\Sigma_1 + \Sigma_2) w$。

In [ ]:
# ============================================================
# 代码 4.3: CSP 算法实现与演示
# ============================================================
import numpy as np
from scipy.linalg import eigh
import matplotlib.pyplot as plt


class CSP:
    """
    Common Spatial Patterns (CSP) 特征提取器。
    
    经典的二分类空域滤波算法，广泛应用于运动想象 BCI。
    提取使两类方差差异最大化的空间滤波器。
    """
    
    def __init__(self, n_components=4):
        """
        Parameters
        ----------
        n_components : int
            保留的 CSP 成分数（前 n/2 和后 n/2 各取一半）
        """
        self.n_components = n_components
        self.filters = None
        self.patterns = None
    
    def fit(self, X_class1, X_class2):
        """
        训练 CSP 滤波器。
        
        Parameters
        ----------
        X_class1 : ndarray (n_trials, n_channels, n_samples)
            类别 1 的 trial 数据
        X_class2 : ndarray (n_trials, n_channels, n_samples)
            类别 2 的 trial 数据
        """
        n_channels = X_class1.shape[1]
        
        # 计算平均协方差矩阵
        cov1 = np.mean([np.cov(trial) for trial in X_class1], axis=0)
        cov2 = np.mean([np.cov(trial) for trial in X_class2], axis=0)
        
        # 正则化（避免奇异）
        cov1 += 1e-6 * np.eye(n_channels)
        cov2 += 1e-6 * np.eye(n_channels)
        
        # 解广义特征值问题
        eigvals, eigvecs = eigh(cov1, cov1 + cov2)
        
        # 按特征值排序（降序）
        idx = np.argsort(eigvals)[::-1]
        eigvecs = eigvecs[:, idx]
        
        # 保留前 n/2 和后 n/2 个特征向量
        half = self.n_components // 2
        selected = np.hstack([eigvecs[:, :half], eigvecs[:, -half:]])
        
        self.filters = selected.T  # (n_components, n_channels)
        self.patterns = np.linalg.pinv(selected).T  # 空间模式（用于可视化）
        
        return self
    
    def transform(self, X):
        """
        应用 CSP 滤波器提取特征。
        
        Parameters
        ----------
        X : ndarray (n_trials, n_channels, n_samples)
            
        Returns
        -------
        features : ndarray (n_trials, n_components)
            CSP 特征 = log(variance of each filtered signal)
        """
        features = []
        for trial in X:
            filtered = self.filters @ trial  # (n_components, n_samples)
            # 特征 = log(方差)
            feat = np.log(np.var(filtered, axis=1) + 1e-10)
            features.append(feat)
        return np.array(features)


# ---- 生成模拟运动想象数据 ----
rng = np.random.default_rng(42)
n_channels = 8
n_trials = 100
n_samples = 250  # 1 秒 @ 250 Hz

# 模拟 CSP 模式：左侧 MI → C3 alpha ERD；右侧 MI → C4 alpha ERD
# 使用简化的空间模式（实际数据中模式来自真实神经源）

def generate_mi_trials(n_trials, active_ch, n_channels=8, n_samples=250):
    """
    生成运动想象 trial 数据。
    
    active_ch: 激活的通道（alpha 抑制更强）
    """
    trials = np.zeros((n_trials, n_channels, n_samples))
    t = np.arange(n_samples) / 250.0
    
    for tr in range(n_trials):
        for ch in range(n_channels):
            # 空间衰减模式：离 active_ch 越近，alpha 抑制越强
            distance = abs(ch - active_ch)
            alpha_amp = 30 * np.exp(-distance / 2)  # 高斯衰减
            beta_amp = 10 * np.exp(-distance / 3)
            
            noise = rng.normal(0, 3, n_samples)
            alpha = alpha_amp * np.sin(2 * np.pi * 10 * t + rng.uniform(0, 2*np.pi))
            beta = beta_amp * np.sin(2 * np.pi * 22 * t + rng.uniform(0, 2*np.pi))
            
            trials[tr, ch] = alpha + beta + noise
    
    return trials


# 类别 1: 左侧 MI (C3=CH2, index 2)
X_left = generate_mi_trials(n_trials, active_ch=2)
# 类别 2: 右侧 MI (C4=CH3, index 3)
X_right = generate_mi_trials(n_trials, active_ch=3)

# 训练 CSP
csp = CSP(n_components=4)
csp.fit(X_left, X_right)

# 提取特征
features_left = csp.transform(X_left)
features_right = csp.transform(X_right)

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. CSP 模式（地形图）
im1 = axes[0].imshow(csp.patterns[:, :4].T, aspect="auto", cmap="RdBu_r",
                     vmin=-0.5, vmax=0.5)
axes[0].set_xlabel("CSP Component")
axes[0].set_ylabel("Channel")
axes[0].set_title("CSP Patterns (Topography)")
axes[0].set_yticks(range(n_channels))
axes[0].set_yticklabels([f"CH{i+1}" for i in range(n_channels)])
axes[0].set_xticks(range(4))
axes[0].set_xticklabels(["C1", "C2", "C3", "C4"])
plt.colorbar(im1, ax=axes[0])

# 2. 特征分布（前 2 个 CSP 成分）
axes[1].scatter(features_left[:, 0], features_left[:, 1],
                c="steelblue", alpha=0.6, label="Left MI", s=30)
axes[1].scatter(features_right[:, 0], features_right[:, 1],
                c="coral", alpha=0.6, label="Right MI", s=30)
axes[1].set_xlabel("CSP Feature 1")
axes[1].set_ylabel("CSP Feature 2")
axes[1].set_title("CSP Feature Space (1st vs 2nd component)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. 单 trial 原始信号对比
t_trial = np.arange(n_samples) / SFREQ
axes[2].plot(t_trial, X_left[0, 2], color="steelblue", label="Left MI (C3)", linewidth=0.8)
axes[2].plot(t_trial, X_right[0, 3], color="coral", label="Right MI (C4)", linewidth=0.8)
axes[2].set_xlabel("Time (s)")
axes[2].set_ylabel("Amplitude (muV)")
axes[2].set_title("Single Trial: C3 (Left MI) vs C4 (Right MI)")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("CSP 特征提取结果:")
print(f"  Left MI  - Feature 1 mean: {features_left[:, 0].mean():.3f}")
print(f"  Right MI - Feature 1 mean: {features_right[:, 0].mean():.3f}")
print(f"  Feature 1 区分度 (Cohen's d): "
      f"{(features_right[:,0].mean() - features_left[:,0].mean()) / "
      f"max(np.std([features_right[:,0], features_left[:,0]]), 1e-6):.2f}")

## 4.4 多通道频谱分析与地形图

将各通道的频带功率映射到头皮上，称为 EEG 地形图 (Topographic Map)。
这是临床 EEG 和 BCI 研究中的标准可视化工具。

In [ ]:
# ============================================================
# 代码 4.4: PSD 多通道对比与频段能量分布
# ============================================================
from utils.helpers import compute_band_powers, BAND_COLORS, CYTON_8CH_1020_MAP

# 使用前面的 eyes_closed 数据
# 逐通道计算频带相对功率
band_names = ["delta", "theta", "alpha", "beta", "gamma"]
bands = {"delta": (0.5, 4), "theta": (4, 8), "alpha": (8, 13),
         "beta": (13, 30), "gamma": (30, 45)}

n_ch = eyes_closed.shape[0]
channel_rel_powers = np.zeros((n_ch, len(band_names)))

for ch in range(n_ch):
    powers = compute_band_powers(eyes_closed[ch:ch+1], SFREQ, bands)
    for j, band in enumerate(band_names):
        channel_rel_powers[ch, j] = powers[band]["relative"] * 100

# 热力图：通道 × 频段
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左: 热力图
im = axes[0].imshow(channel_rel_powers.T, aspect="auto", cmap="YlOrRd",
                     vmin=0, vmax=channel_rel_powers.max() * 0.9)
axes[0].set_xlabel("Channel")
axes[0].set_ylabel("Frequency Band")
axes[0].set_title("Channel × Band Relative Power (%)")
axes[0].set_xticks(range(n_ch))
axes[0].set_xticklabels([CYTON_8CH_1020_MAP.get(i+1, f"CH{i+1}")
                          for i in range(n_ch)], rotation=45)
axes[0].set_yticks(range(len(band_names)))
axes[0].set_yticklabels(band_names)
plt.colorbar(im, ax=axes[0], label="Relative Power (%)")

# 右: 堆叠柱状图
x = np.arange(n_ch)
bottom = np.zeros(n_ch)
for j, band in enumerate(band_names):
    axes[1].bar(x, channel_rel_powers[:, j], bottom=bottom,
                color=BAND_COLORS[band], label=band, alpha=0.85)
    bottom += channel_rel_powers[:, j]

axes[1].set_xlabel("Channel")
axes[1].set_ylabel("Relative Power (%)")
axes[1].set_title("Stacked Band Power per Channel")
axes[1].set_xticks(x)
axes[1].set_xticklabels([CYTON_8CH_1020_MAP.get(i+1, f"CH{i+1}")
                          for i in range(n_ch)], rotation=45)
axes[1].legend(fontsize=8)
axes[1].set_ylim(0, np.sum(channel_rel_powers, axis=1).max() * 1.1)

plt.tight_layout()
plt.show()

print("通道-频段相对功率矩阵 (%):")
header = f"{'Channel':<10}" + "".join([f"{b:>8}" for b in band_names])
print(header)
print("-" * (10 + 8 * len(band_names)))
for ch in range(n_ch):
    name = CYTON_8CH_1020_MAP.get(ch+1, f"CH{ch+1}")
    vals = "".join([f"{channel_rel_powers[ch, j]:>8.2f}" for j in range(len(band_names))])
    print(f"{name:<10}{vals}")

## 单元小结

本单元涵盖：
1. 频带功率（绝对和相对）的计算与两种状态对比
2. ERD/ERS 分析流程：基线计算 → 时频功率 → 百分比变化
3. CSP 空域滤波器的数学原理、代码实现和特征提取
4. 多通道频谱分析：通道×频段热力图与堆叠柱状图

**下一步：** Unit 5 将结合 CSP 特征和分类器构建完整的 BCI 系统。